# VGGNet — Very Deep Convolutional Networks for Large-Scale Image Recognition

**Citation:** Simonyan, K., & Zisserman, A. (2015). Very deep convolutional networks for large-scale image recognition. In *International Conference on Learning Representations (ICLR 2015)*. arXiv:1409.1556.

**Registered names:** `vgg11`, `vgg13`, `vgg16`, `vgg19`, `vgg11_bn`, `vgg13_bn`, `vgg16_bn`, `vgg19_bn`

**Summary:** VGGNet's central finding: depth with small 3×3 filters outperforms shallower networks with larger kernels. Two stacked 3×3 convolutions have the same receptive field as one 5×5 but fewer parameters and more non-linearity.

---

| Model | Layers | Params | Top-1 |
|-------|--------|--------|-------|
| VGG-11 | 11 | 133M | 69.0% |
| VGG-16 | 16 | 138M | 71.5% |
| VGG-19 | 19 | 144M | 71.1% |
| VGG-16-BN | 16 | 138M | 73.4% |

## Architecture Diagram

```
Input [B, 3, 224, 224]
       │
       ▼
╔══════════════════════╗
║ Stage 0             ║  1-2 × Conv(64, 3×3) + MaxPool  → [B, 64,  112, 112]
╚══════════════════════╝
       │
╔══════════════════════╗
║ Stage 1             ║  1-2 × Conv(128, 3×3) + MaxPool → [B, 128,  56,  56]
╚══════════════════════╝
       │
╔══════════════════════╗
║ Stage 2             ║  2-4 × Conv(256, 3×3) + MaxPool → [B, 256,  28,  28]
╚══════════════════════╝
       │
╔══════════════════════╗
║ Stage 3             ║  2-4 × Conv(512, 3×3) + MaxPool → [B, 512,  14,  14]
╚══════════════════════╝
       │
╔══════════════════════╗
║ Stage 4             ║  2-4 × Conv(512, 3×3) + MaxPool → [B, 512,   7,   7]
╚══════════════════════╝
       │
  AdaptiveAvgPool(7×7) → Flatten
  FC(25088→4096) → ReLU → Dropout
  FC(4096→4096)  → ReLU → Dropout
       │
       ▼
pooled_output [B, 4096]
```

**VGG-16 configuration:** `[64,64,M, 128,128,M, 256,256,256,M, 512,512,512,M, 512,512,512,M]`
where M = MaxPool.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(15, 8))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')
ax.set_xlim(0, 15); ax.set_ylim(0, 8); ax.axis('off')

# VGG block counts per variant
models = {
    'VGG-11': [1, 1, 2, 2, 2],
    'VGG-13': [2, 2, 2, 2, 2],
    'VGG-16': [2, 2, 3, 3, 3],
    'VGG-19': [2, 2, 4, 4, 4],
}
ch_labels = ['64', '128', '256', '512', '512']
colors = ['#4a9eff', '#6bcb77', '#ff9f43', '#ff6b9d', '#a29bfe']

# Draw comparison: each column = a model variant
for col, (name, blocks) in enumerate(models.items()):
    x_base = 1.5 + col * 3.2
    ax.text(x_base + 0.6, 7.5, name, ha='center', color='white', fontsize=11, fontweight='bold')
    y = 6.5
    for stage_i, (n, color) in enumerate(zip(blocks, colors)):
        for j in range(n):
            bx = x_base + j * 0.28
            box = FancyBboxPatch((bx, y - 0.3), 0.25, 0.55,
                                  boxstyle='round,pad=0.04', facecolor=color,
                                  edgecolor='white', linewidth=0.6, alpha=0.85)
            ax.add_patch(box)
            ax.text(bx + 0.125, y + 0.0, f'C{ch_labels[stage_i]}', ha='center',
                    va='center', fontsize=5.5, color='white')
        # MaxPool
        mp_x = x_base + n * 0.28
        ax.add_patch(FancyBboxPatch((mp_x, y - 0.25), 0.22, 0.45,
                                     boxstyle='round,pad=0.03', facecolor='#333344',
                                     edgecolor='#aaaaaa', linewidth=0.6, alpha=0.9))
        ax.text(mp_x + 0.11, y, 'MP', ha='center', va='center', fontsize=5, color='#cccccc')
        y -= 1.15

ax.text(7.5, 7.8, 'VGG Family — Architecture Comparison', ha='center',
        color='white', fontsize=13, fontweight='bold')

# Legend
for i, (name, color) in enumerate(zip([f'Stage {s}' for s in range(5)], colors)):
    ax.add_patch(FancyBboxPatch((0.3 + i * 2.8, 0.2), 0.35, 0.35,
                                 boxstyle='round,pad=0.04', facecolor=color, edgecolor='none'))
    ax.text(0.75 + i * 2.8, 0.38, name, color='white', fontsize=8, va='center')

plt.tight_layout()
plt.show()

## Key Innovations

### 1. Small Filters — The 3×3 Principle
VGGNet's central argument: two stacked 3×3 convolutions are equivalent to one 5×5 conv in terms of receptive field:
$$r_2 \times r_2 \text{ via stack of 2: } 3+(3-1)=5$$
Three stacked 3×3 convolutions = one 7×7 conv. But the stacked version has:
- **Fewer parameters:** $3 \times (3^2 C^2) = 27C^2$ vs $7^2 C^2 = 49C^2$
- **More non-linearity:** 3 ReLU layers vs 1

### 2. Monotonic Channel Doubling
Each stage doubles channels (64→128→256→512), capturing progressively more abstract features.

### 3. Homogeneous Design
No special modules or hyperparameters per stage — the same pattern (3×3 Conv + ReLU) repeated. This simplicity made VGGNet very influential as a transfer learning backbone.

### 4. Large FC Head
The 3-layer FC head (4096→4096→1000) holds most of the parameters (102M/138M). Later work (GAP + small head) reduced this dramatically.

### BatchNorm Variants
Adding BatchNorm after each Conv (not in the original paper) significantly improves training stability and accuracy:
$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta$$

## Code Walkthrough

Implemented in `hale_core/nn/vision/cnn/vgg.py`.

**Configuration-driven stage builder:**
```python
_CFGS = {
    'vgg16': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'],
}

def _make_vgg_stages(cfg, batch_norm, in_chans=3):
    stages, layers, in_ch = [], [], in_chans
    for v in cfg:
        if v == 'M':
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            stages.append(nn.Sequential(*layers))
            layers = []
        else:
            layers.append(nn.Conv2d(in_ch, v, kernel_size=3, padding=1))
            if batch_norm: layers.append(nn.BatchNorm2d(v))
            layers.append(nn.ReLU(inplace=True))
            in_ch = v
    return stages
```

All 8 variants (vgg11/13/16/19, ±BN) share one `VGG` class. The `_bn` variants pass `batch_norm=True`.

In [ ]:
import torch
import sys
sys.path.insert(0, '../../src')

from hale_core.nn.vision.registry import build_vision_backbone

for name in ['vgg11', 'vgg16', 'vgg19', 'vgg16_bn']:
    bb = build_vision_backbone(name, img_size=224)
    bb.eval()
    x = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        out = bb(x)
    fmap_shapes = [tuple(f.shape) for f in out.feature_maps]
    print(f'{name:12s} pooled={tuple(out.pooled_output.shape)} '
          f'params={bb.num_parameters()/1e6:.1f}M')
    print(f'             feature_maps: {fmap_shapes}')

## References

| Paper | Venue | Key Contribution |
|-------|-------|------------------|
| Simonyan & Zisserman (2015). Very Deep CNNs. | ICLR 2015 | VGGNet — small filter depth principle |
| Krizhevsky et al. (2012). ImageNet Classification. | NeurIPS 2012 | AlexNet — predecessor, large filters |
| He et al. (2016). Deep Residual Learning. | CVPR 2016 | ResNet — skip connections supersede VGG |
| Ioffe & Szegedy (2015). Batch Normalisation. | ICML 2015 | BN — enables _bn variants |
| Zeiler & Fergus (2014). Visualising CNNs. | ECCV 2014 | Analysis of why depth helps |

**Full Citation:**
> Simonyan, K., & Zisserman, A. (2015). Very deep convolutional networks for large-scale image recognition. In *International Conference on Learning Representations (ICLR 2015)*. arXiv:1409.1556.